# NER task

Training an SVM on CoNLL-2003 and testing it on the project NER test set.
Features are the word + its POS tag, like in lab 4. Test set only has the tokens,
so I pos-tag those with nltk first.

In [1]:
import csv
import nltk
from sklearn.feature_extraction import DictVectorizer
from sklearn.svm import LinearSVC
from sklearn.metrics import classification_report

nltk.download('averaged_perceptron_tagger_eng', quiet=True)
nltk.download('punkt', quiet=True)

True

### paths

In [2]:
# conll train file from the course repo (lab4/CONLL2003/CONLL2003/train.txt)
CONLL_TRAIN = "train.txt"
NER_TEST    = "NER-test.tsv"

### load training data (conll)
format per line: word  pos  chunk  ner . blank line = end of sentence.

In [3]:
def read_conll(path):
    feats = []
    labels = []
    for line in open(path, encoding="utf-8"):
        line = line.strip()
        if line == "" or line.startswith("-DOCSTART-"):
            continue
        cols = line.split()
        if len(cols) >= 4:
            word = cols[0]
            pos  = cols[1]
            ner  = cols[3]
            feats.append({"word": word, "pos": pos})
            labels.append(ner)
    return feats, labels

train_feats, train_labels = read_conll(CONLL_TRAIN)
print("training tokens:", len(train_feats))
print("example:", train_feats[0], "->", train_labels[0])

training tokens: 203621
example: {'word': 'EU', 'pos': 'NNP'} -> B-ORG


In [4]:
from collections import Counter

# how often each label appears in the training data (and how balanced it is)
counts = Counter(train_labels)
total = len(train_labels)
print("label distribution in training data:")
for lab, n in counts.most_common():
    print(f"  {lab:7} {n:6}  ({100*n/total:.1f}%)")

label distribution in training data:
  O       169578  (83.3%)
  B-LOC     7140  (3.5%)
  B-PER     6600  (3.2%)
  B-ORG     6321  (3.1%)
  I-PER     4528  (2.2%)
  I-ORG     3704  (1.8%)
  B-MISC    3438  (1.7%)
  I-LOC     1157  (0.6%)
  I-MISC    1155  (0.6%)


### load test set
group tokens back into sentences (by sentence id) so pos tagging makes sense.

In [5]:
sentences = {}
with open(NER_TEST, encoding="utf-8") as f:
    reader = csv.reader(f, delimiter="\t")
    next(reader)  # header
    for row in reader:
        if len(row) < 4:
            continue
        sid = row[0].strip()
        token = row[2].strip()
        tag = row[3].strip().replace("\r", "")
        if sid not in sentences:
            sentences[sid] = {"tokens": [], "tags": []}
        sentences[sid]["tokens"].append(token)
        sentences[sid]["tags"].append(tag)

print("sentences in test:", len(sentences))

sentences in test: 10


In [6]:
test_feats = []
test_labels = []
for sid in sorted(sentences, key=lambda x: int(x)):
    tokens = sentences[sid]["tokens"]
    tags = sentences[sid]["tags"]
    pos_tags = [p for (w, p) in nltk.pos_tag(tokens)]   # pos tag the sentence
    for w, p, t in zip(tokens, pos_tags, tags):
        test_feats.append({"word": w, "pos": p})
        test_labels.append(t)

print("test tokens:", len(test_feats))

test tokens: 214


### vectorize + train svm
put train+test together for the vectorizer so the columns match, then split back.

In [7]:
# fit the vectorizer on train+test together so both share the same feature columns
# (this only lines up the one-hot columns; the test LABELS are never used in training)
vec = DictVectorizer()
all_feats = train_feats + test_feats
X = vec.fit_transform(all_feats)

split = len(train_feats)
X_train = X[:split]
X_test = X[split:]

clf = LinearSVC(max_iter=5000)
clf.fit(X_train, train_labels)
pred = clf.predict(X_test)
print("done")

done


### results

In [8]:
labels = sorted(set(test_labels) - {"O"}) + ["O"]
print(classification_report(test_labels, pred, labels=labels, zero_division=0))

              precision    recall  f1-score   support

       B-LOC       0.50      0.50      0.50         4
      B-MISC       0.67      0.67      0.67         3
       B-ORG       0.00      0.00      0.00         4
       B-PER       0.75      0.50      0.60         6
       I-LOC       0.67      1.00      0.80         2
      I-MISC       0.00      0.00      0.00         1
       I-ORG       0.50      0.67      0.57         3
       I-PER       0.64      0.88      0.74         8
           O       0.99      1.00      1.00       183

    accuracy                           0.94       214
   macro avg       0.52      0.58      0.54       214
weighted avg       0.93      0.94      0.93       214



### look at the entity tokens (for error analysis)
only print tokens where gold or prediction is not O.

In [9]:
words = [f["word"] for f in test_feats]
for w, g, p in zip(words, test_labels, pred):
    if g != "O" or p != "O":
        mark = "" if g == p else "   wrong"
        print(f"{w:15} gold={g:7} pred={p:7}{mark}")

Warner          gold=B-ORG   pred=I-PER     wrong
Brothers        gold=I-ORG   pred=I-ORG  
New             gold=B-ORG   pred=B-LOC     wrong
York            gold=I-ORG   pred=I-LOC     wrong
University      gold=I-ORG   pred=I-ORG  
Soho            gold=B-LOC   pred=I-PER     wrong
Italian         gold=B-MISC  pred=B-MISC 
Jane            gold=B-PER   pred=B-PER  
Austen          gold=I-PER   pred=I-PER  
Carl            gold=B-PER   pred=B-PER  
Brashear        gold=I-PER   pred=I-PER  
Cuba            gold=B-PER   pred=B-LOC     wrong
Gooding         gold=I-PER   pred=I-PER  
Jr.             gold=I-PER   pred=I-PER  
African         gold=B-MISC  pred=I-MISC    wrong
American        gold=I-MISC  pred=B-MISC    wrong
Navy            gold=B-ORG   pred=I-ORG     wrong
Chris           gold=B-PER   pred=B-PER  
O'Donnell       gold=I-PER   pred=I-PER  
Amsterdam       gold=B-LOC   pred=I-ORG     wrong
Blauwbrug       gold=B-ORG   pred=I-PER     wrong
Dame            gold=B-PER   pred=I-PE